In [2]:
import pandas as pd 
import requests
from pathlib import Path 
import zipfile

In [7]:
URL = "https://goldencopy.gleif.org/api/v2/golden-copies/publishes/lei2/latest.csv"
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
output_path = PROJECT_ROOT / "data" / "gleif_latest.zip"

In [9]:
if output_path.exists():
    print("File already exists")
else:
    url_request = requests.get(URL, stream=True)
    output_path.parent.mkdir(parents=True, exist_ok=True)
    with open(output_path, "wb") as f:
        for chunk in url_request.iter_content(chunk_size=8192):
            f.write(chunk)
    print(f"Downloaded to {output_path}")

File already exists


In [10]:
print(output_path)
print(output_path.exists())
print(f"{output_path.stat().st_size / 1024**2:.1f} MB")

/Users/katerina.ale/Projects/entity-resolution-gleif/data/gleif_latest.zip
True
457.2 MB


In [11]:
with zipfile.ZipFile(output_path) as zf:
    zip_contents = zf.namelist()

print(zip_contents)

['20260421-0800-gleif-goldencopy-lei2-golden-copy.csv']


In [16]:
#### csv_filename = zip_contents[0]
csv_path = PROJECT_ROOT / "data" / csv_filename

if csv_path.exists():
    print("Zip already extracted, skipping")
else: 
    print("Extracting...") 

    with zipfile.ZipFile(output_path) as zf:
        with zf.open(csv_filename) as csv_stream:
            df = pd.read_csv(csv_stream, nrows=100_000)

    print(df.shape)
    print(df.columns.tolist()[:10])
    df.head()

Extracting...


/var/folders/tl/j44t818d1hscbqxqn8t3p7x40000gn/T/ipykernel_40124/2364906972.py:11: DtypeWarning: Columns (0: Entity.OtherEntityNames.OtherEntityName.3, 1: Entity.OtherEntityNames.OtherEntityName.3.xmllang, 2: Entity.OtherEntityNames.OtherEntityName.3.type, 3: Entity.OtherEntityNames.OtherEntityName.4, 4: Entity.OtherEntityNames.OtherEntityName.4.xmllang, 5: Entity.OtherEntityNames.OtherEntityName.4.type, 6: Entity.OtherEntityNames.OtherEntityName.5, 7: Entity.OtherEntityNames.OtherEntityName.5.xmllang, 8: Entity.OtherEntityNames.OtherEntityName.5.type, 9: Entity.TransliteratedOtherEntityNames.TransliteratedOtherEntityName.2, 10: Entity.TransliteratedOtherEntityNames.TransliteratedOtherEntityName.2.xmllang, 11: Entity.TransliteratedOtherEntityNames.TransliteratedOtherEntityName.2.type, 12: Entity.LegalAddress.AddressNumber, 13: Entity.LegalAddress.AddressNumberWithinBuilding, 14: Entity.LegalAddress.MailRouting, 15: Entity.LegalAddress.AdditionalAddressLine.1, 16: Entity.LegalAddress.Ad

(100000, 338)
['LEI', 'Entity.LegalName', 'Entity.LegalName.xmllang', 'Entity.OtherEntityNames.OtherEntityName.1', 'Entity.OtherEntityNames.OtherEntityName.1.xmllang', 'Entity.OtherEntityNames.OtherEntityName.1.type', 'Entity.OtherEntityNames.OtherEntityName.2', 'Entity.OtherEntityNames.OtherEntityName.2.xmllang', 'Entity.OtherEntityNames.OtherEntityName.2.type', 'Entity.OtherEntityNames.OtherEntityName.3']


In [17]:
df_slim = df[["LEI", "Entity.LegalName", "Entity.LegalName.xmllang", "Entity.LegalJurisdiction"]].rename(
    columns={
        "Entity.LegalName": "legal_name",
        "Entity.LegalName.xmllang": "lang",
        "Entity.LegalJurisdiction": "jurisdiction",
    }
)
print(df_slim.shape)
df_slim.head()

(100000, 4)


,LEI,legal_name,lang,jurisdiction
0,001GPB6A9XPE8XJICC14,Fidelity Advisor Leveraged Company Stock Fund,en,US
1,004L5FPTUREIWK9T2N63,"Hutchin Hill Capital, LP",en,US-DE
2,00EHHQ2ZHDCFXJCPCL46,Vanguard Fiduciary Trust Company Vanguard Russ...,en,US-PA
3,00GBW0Z2GYIER7DHDS71,"ARISTEIA CAPITAL, L.L.C.",en,US-DE
4,00KLB2PFTM3060S2N216,Oakmark International Fund,en,US


In [18]:
df_slim["lang"].value_counts()

lang
en    81357
sk     4890
da     2920
el     2179
no     1284
      ...  
mh        1
uz        1
sr        1
mn        1
az        1
Name: count, Length: 87, dtype: int64

In [19]:
df_slim["jurisdiction"].value_counts().head(20)

jurisdiction
GB       55168
SK        4862
DK        3935
LU        3364
JE        3006
GR        2229
VG        2218
IE        1898
GG        1767
KY        1670
CY        1659
NO        1440
IM        1072
SE         965
US-DE      809
AU         730
FR         643
DE         632
MT         622
NG         618
Name: count, dtype: int64

In [20]:
df_slim["legal_name"].isna().sum()

np.int64(0)